# 03 — Building vs. Housing Analysis

**Owned by**: Stage 7 (documentation deliverable, per `PROJECT_BLUEPRINT.md`'s Stage 7 section and
`docs/risk_register.md` R-003).

Shows visually, rather than only asserting in prose, why ACS housing-unit counts are excluded from
every denominator in this project's coverage-gap formula (`src/gaps.py`) — the formula compares
Overture against Microsoft building **footprints**, never against ACS housing units.

## 1. Setup

Same repo-root-detection and import style as `01_eda.ipynb`/`02_pipeline_dev.ipynb`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src").exists(), (
    f"Could not locate the repository root (a directory containing 'src/') from {Path.cwd()!r}. "
    "Run this notebook from the repository root or from notebooks/, its only supported locations."
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# Deliberately NOT adding REPO_ROOT / "src" itself to sys.path (unlike 01_eda.ipynb, which gets
# away with it only because it happens to import geopandas before that insertion runs): src/
# contains its own statistics.py, and putting src/ directly on sys.path would shadow the stdlib
# `statistics` module the moment anything imports geopandas (geopandas.explore does `from
# statistics import mean` internally) -- causing a confusing ImportError far from this cell. Every
# import below is package-qualified (`from src.config import ...`), which only needs REPO_ROOT.

from src.config import LAYER_CENSUS_ACS_HOUSING, REGIONS, reference_s3_path  # noqa: E402
from src.io import _assert_geoid_is_string as assert_geoid_is_string  # noqa: E402
from src.io import _s3_filesystem as s3_filesystem  # noqa: E402

pd.set_option("display.max_columns", 60)
print(f"Repository root: {REPO_ROOT}")

## 2. Load ACS housing units per tract

`census-acs-housing` has no registered pandera schema (`docs/data_manifest.md` Section 7), so this
reuses `01_eda.ipynb` Section 3's exact raw-loader pattern rather than inventing a second one — the
same temporary, explicitly-flagged exception to `load_reference_layer`, applying the same two
standing guards (GEOID-string dtype; geometry is dropped here since only `housing_units` is
needed).

In [ ]:
import pyarrow.parquet as pq

_fs = s3_filesystem()

housing_frames = []
for region in REGIONS:
    path = reference_s3_path(region, LAYER_CENSUS_ACS_HOUSING)
    table = pq.read_table(path, filesystem=_fs, columns=["GEOID", "housing_units"])
    df = table.to_pandas()
    assert_geoid_is_string(df, context=f"census-acs-housing[{region}]")
    df["region"] = region
    housing_frames.append(df)

housing = pd.concat(housing_frames, ignore_index=True)
print(f"Loaded housing_units for {len(housing)} tracts across {housing['region'].nunique()} regions.")
housing.head()

## 3. Load building footprint counts per tract

From Stage 6's already-materialized `data/processed/<region>-tract-features.parquet` — no raw
Overture/Microsoft read needed here, these columns already exist.

In [ ]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

feature_frames = []
for region in REGIONS:
    df = pd.read_parquet(
        PROCESSED_DIR / f"{region}-tract-features.parquet",
        columns=["GEOID", "region", "overture_building_count_centroid", "microsoft_building_count_centroid"],
    )
    feature_frames.append(df)

buildings = pd.concat(feature_frames, ignore_index=True)
print(f"Loaded building counts for {len(buildings)} tracts.")
buildings.head()

## 4. Join and compare

A tract's ACS `housing_units` count and its Overture/Microsoft building-footprint counts are
structurally different quantities, not just numerically different ones:

- `housing_units` counts **dwelling units** — a single apartment building with 200 units is 1
  building footprint but 200 housing units.
- Overture/Microsoft building counts count **footprints** — a single-family house, a 200-unit
  apartment block, a warehouse, a church, and a strip mall are each exactly 1 footprint.

If housing units were used as `building_gap`'s denominator, a tract with dense multi-unit housing
(e.g. an urban core) would show a large apparent "gap" purely from this unit/footprint mismatch,
with zero relationship to whether Overture's map data is actually missing buildings that exist.
This is a structural confound, not a coverage-measurement signal — visualized below, not just
asserted.

In [ ]:
joined = buildings.merge(housing[["GEOID", "housing_units"]], on="GEOID", how="inner")
print(f"Joined {len(joined)} of {len(buildings)} tracts ({len(joined) / len(buildings):.1%}).")

joined["housing_to_building_ratio"] = joined["housing_units"] / joined["overture_building_count_centroid"].replace(0, pd.NA)

summary = joined.groupby("region")[["housing_units", "overture_building_count_centroid", "housing_to_building_ratio"]].median()
summary.columns = ["median_housing_units", "median_overture_buildings", "median_housing_units_per_building"]
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for region, group in joined.groupby("region"):
    ax.scatter(group["overture_building_count_centroid"], group["housing_units"], s=8, alpha=0.4, label=region)
lims = [0, joined[["overture_building_count_centroid", "housing_units"]].quantile(0.98).max()]
ax.plot(lims, lims, "k--", linewidth=1, label="1:1 (what a clean denominator would look like)")
ax.set_xlabel("Overture building footprint count (per tract)")
ax.set_ylabel("ACS housing units (per tract)")
ax.set_title("Building footprints vs. housing units, per tract")
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(joined["housing_to_building_ratio"].dropna().clip(upper=joined["housing_to_building_ratio"].quantile(0.98)), bins=40)
ax.axvline(1.0, color="k", linestyle="--", linewidth=1, label="1:1 (one housing unit per footprint)")
ax.set_xlabel("Housing units per building footprint")
ax.set_ylabel("Tract count")
ax.set_title("Distribution of the housing-unit-to-footprint ratio")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / "docs" / "figures" / "03_building_vs_housing_ratio.png", dpi=120)
plt.show()

## 5. Conclusion

The housing-unit-to-footprint ratio is not clustered near 1:1 — it varies widely both within and
across regions (see `summary` above and the histogram's spread), exactly because the two quantities
measure different things (dwelling units vs. building footprints). Using `housing_units` as a
coverage-gap denominator would conflate housing density with mapping completeness, both of which
vary independently and for unrelated reasons across urban/rural and tribal/non-tribal tracts —
precisely the kind of confound `docs/risk_register.md` R-003 was written to catch before it could
silently bias the formula. `src/gaps.py`'s `building_gap` compares Overture against Microsoft
building footprints exclusively — never against ACS housing units — and this notebook is the
visual confirmation that this was the right call, not just an assumption. R-003 is closed on this
basis; see `docs/decision_log.md` and `docs/risk_register.md` for the corresponding entries.